In [1]:
# ============================================================
# Cell 1: Ablation Study — Model Component Analysis
# Purpose: Compare individual contributions of CNN and Attention layers
#          against the full BanglaBERT + CNN + Attention model
# ============================================================

import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from tqdm.auto import tqdm

PROJECT_ROOT = "/mnt/g/banglafake-detection"
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "csebuetnlp/banglabert"
MAX_LENGTH = 256

print("Ablation Study Initiated")
print("Device:", device)
print("Base Model:", MODEL_NAME)
print("Components to test:")
print("- BanglaBERT only")
print("- BanglaBERT + CNN only") 
print("- BanglaBERT + Attention only")
print("- BanglaBERT + CNN + Attention (Full)")

Ablation Study Initiated
Device: cuda
Base Model: csebuetnlp/banglabert
Components to test:
- BanglaBERT only
- BanglaBERT + CNN only
- BanglaBERT + Attention only
- BanglaBERT + CNN + Attention (Full)


In [2]:
# ============================================================
# Cell 2: Define Individual Model Architectures for Ablation
# Purpose: Create 4 separate model variations to compare:
#          - BERT-only baseline
#          - BERT + CNN (w/o Attention)
#          - BERT + Attention (w/o CNN)  
#          - BERT + CNN + Attention (full model)
# ============================================================

class BanglaBERTOnly(nn.Module):
    """Baseline: Only the pretrained BERT encoder followed by a linear classifier"""
    def __init__(self, model_name, num_classes=2, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output  # Use CLS token
        output = self.dropout(pooled_output)
        logits = self.classifier(output)
        return logits

class BanglaBERTWithCNN(nn.Module):
    """BERT + CNN only (no attention mechanism)"""
    def __init__(self, model_name, cnn_channels=256, kernel_size=3, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.cnn = nn.Conv1d(
            in_channels=self.bert.config.hidden_size,
            out_channels=cnn_channels,
            kernel_size=kernel_size,
            padding=kernel_size // 2
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(cnn_channels, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Shape: [batch, seq_len, hidden_size]
        hidden_states = outputs.last_hidden_state
        
        # Apply mask before CNN to ignore padding
        mask = attention_mask.unsqueeze(-1).float()  # [batch, seq_len, 1]
        masked_hidden = hidden_states * mask
        # Transpose for CNN: [batch, hidden, seq_len]
        transposed = masked_hidden.permute(0, 2, 1)
        # Apply CNN
        cnn_output = torch.relu(self.cnn(transposed))  # [batch, cnn_channels, seq_len]
        # Pool across sequence length
        pooled = torch.mean(cnn_output, dim=2)  # [batch, cnn_channels]
        dropped_out = self.dropout(pooled)
        logits = self.classifier(dropped_out)
        return logits

class BanglaBERTWithAttention(nn.Module):
    """BERT + Attention only (no CNN)"""
    def __init__(self, model_name, attention_dim=128, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        # Simple attention projection
        self.attention_projection = nn.Linear(self.bert.config.hidden_size, attention_dim)
        self.attention_score = nn.Linear(attention_dim, 1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)  # Direct from BERT

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state  # [batch, seq, hidden]

        # Apply mask to padding tokens
        mask = attention_mask.float()  # [batch, seq_len]
        
        # Compute attention weights
        attention_hidden = torch.tanh(self.attention_projection(hidden_states))  # [batch, seq, att_dim]
        attention_logits = self.attention_score(attention_hidden).squeeze(-1)    # [batch, seq]

        # Mask attention scores for padding (very low value)
        attention_logits = attention_logits.masked_fill(attention_mask == 0, -1e9)
        attention_weights = torch.softmax(attention_logits, dim=1)  # [batch, seq]

        # Weighted sum of hidden states
        weighted_hidden = hidden_states * attention_weights.unsqueeze(-1)  # [batch, seq, hidden]
        attended_output = weighted_hidden.sum(dim=1)  # [batch, hidden]

        dropped_out = self.dropout(attended_output)
        logits = self.classifier(dropped_out)
        return logits

class BanglaBERTFullModel(nn.Module):  # This is your original model for comparison
    def __init__(self, model_name, cnn_channels=256, kernel_size=3, attention_dim=128, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        # CNN
        self.cnn = nn.Conv1d(
            in_channels=self.bert.config.hidden_size,
            out_channels=cnn_channels,
            kernel_size=kernel_size,
            padding=kernel_size // 2
        )
        # Attention
        self.attention_projection = nn.Linear(cnn_channels, attention_dim)
        self.attention_score = nn.Linear(attention_dim, 1)
        # Classifier
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(cnn_channels, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        x = outputs.last_hidden_state  # [batch, seq, hidden]

        # Mask padding before CNN
        mask = attention_mask.unsqueeze(-1).float()
        x = x * mask

        # CNN expects [batch, hidden, seq]
        x = x.permute(0, 2, 1)  # -> [batch, hidden, seq]
        x = torch.relu(self.cnn(x))  # -> [batch, channels, seq]
        x = x.permute(0, 2, 1)  # -> [batch, seq, channels]

        # Attention on CNN outputs
        attention_hidden = torch.tanh(self.attention_projection(x))  # [batch, seq, att_dim]
        attention_logits = self.attention_score(attention_hidden).squeeze(-1)  # [batch, seq]

        # Apply mask to attention logits
        attention_logits = attention_logits.masked_fill(attention_mask == 0, -1e9)
        attention_weights = torch.softmax(attention_logits, dim=1)  # [batch, seq]

        # Weighted sum
        attended_output = (x * attention_weights.unsqueeze(-1)).sum(dim=1)  # [batch, channels]
        logits = self.classifier(attended_output)
        return logits

print("Model architectures defined successfully!")
print("- BanglaBERTOnly")
print("- BanglaBERTWithCNN")
print("- BanglaBERTWithAttention")
print("- BanglaBERTFullModel")

Model architectures defined successfully!
- BanglaBERTOnly
- BanglaBERTWithCNN
- BanglaBERTWithAttention
- BanglaBERTFullModel
